# HAM10000 期末實驗 02：EfficientNet-B0（E1、E2、E3、E3b、E4 與彙總）

對應報告參之四、參之五。需要 **T4 GPU**（執行階段 → 變更執行階段類型）。建議先執行 00 與 01，這份最後的彙總會自動合併 Baseline 結果。

**執行內容**
1. **回合數試跑**：種子 42、提出方法、上限 20 回合，提前停止規則與正式訓練相同，只看驗證集，不產生測試集預測。第 12 回合之後的驗證集 macro-F1 若再提升超過 0.01，正式實驗用 20 回合，否則用 12 回合。決定存檔後不再更動。
2. **正式實驗**：每個種子 8 次訓練，共 40 次。
   - 依病灶切分 × 五種不平衡處理（E1、E2）
   - 提出方法 × 依影像切分（E3、E3b 模型 A）
   - 提出方法 × 移除與測試集同病灶的影像（E3b 模型 B）
   - 提出方法 × 移除等量且與測試集無關的病灶（E3b 模型 C）
3. 每次訓練保存驗證集、測試集、外部資料全部 1,705 張的預測機率（附影像編號），以及最佳回合的模型權重（40 次訓練約 650 MB）。
4. **分析**：平均值 ± 標準差、同種子配對差值、黑色素瘤閾值與平均精確率（AP）、E3b 分群、E4 外部驗證、Grad-CAM。

**可續跑**：每完成一次訓練就存到雲端硬碟。斷線後重新「全部執行」會自動略過已完成的部分。每次訓練約 8–15 分鐘，全部約 7–9 小時，可分多次執行。
> **正式版**：以 Claude 原版與 ChatGPT 審查修正版合併。所有預測都附影像編號，分析時依編號對齊。

## 0. 環境設定

In [ ]:
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=False)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE = os.environ.get('HAM_BASE') or ('/content/drive/MyDrive/ham10000_project' if (IN_COLAB and USE_DRIVE) else './ham10000_project')

In [ ]:
import time, json, random, platform, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torchvision
from torchvision import models
from torchvision.transforms import v2 as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay)

# ===== 實驗設定（報告表 3-2、表 3-3、表 3-5）=====
SEEDS = [42, 43, 44, 45, 46]
PILOT_SEED = 42
PILOT_EPOCHS, DEFAULT_EPOCHS = 20, 12
PILOT_MIN_GAIN = 0.01        # 第 12 回合後驗證集 macro-F1 需提升超過此值才改用 20 回合
IMG_SIZE = 224
PATIENCE = 3
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-4
PRETRAINED = os.environ.get('HAM_SMOKE') != '1'   # 程式檢查時不下載預訓練權重
NUM_WORKERS = int(os.environ.get('HAM_WORKERS', 2))   # Windows 本機請設 0
VAL_RATIO, TEST_RATIO = 0.15, 0.15
RUN_E4 = True
SMOKE_TEST = os.environ.get('HAM_SMOKE') == '1'     # 只供程式檢查用，正式執行請保持 False
if SMOKE_TEST:
    IMG_SIZE, PILOT_EPOCHS, DEFAULT_EPOCHS, BATCH_SIZE = 64, 3, 2, 16

RES_DIR = os.path.join(BASE, 'results_final_cnn')
PRED_DIR = os.path.join(RES_DIR, 'predictions')
MODEL_DIR = os.path.join(RES_DIR, 'models')
CACHE_DIR = os.path.join(BASE, 'cache')
CHECK_DIR = os.path.join(BASE, 'data_check')
BASELINE_DIR = os.path.join(BASE, 'results_final_baseline')
for d in [RES_DIR, PRED_DIR, MODEL_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('裝置：', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
if DEVICE == 'cpu' and not SMOKE_TEST:
    print('⚠ 沒有偵測到 GPU，請到「執行階段 → 變更執行階段類型」選 T4 GPU')

CLASS_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLS2IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
MEL = CLS2IDX['mel']
SPLITS = ['lesion', 'image', 'image_noleak', 'image_ctrl']
SPLIT_LABEL = {'lesion': '依病灶切分', 'image': '依影像切分（模型 A）',
               'image_noleak': '移除與測試集同病灶的影像（模型 B）', 'image_ctrl': '移除等量無關病灶（模型 C）'}

## 1. 讀取資料

In [ ]:
DATA_DIR = os.environ.get('HAM_DATA_DIR')
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

meta_path, img_paths = None, {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f == 'HAM10000_metadata.csv' and meta_path is None:
            meta_path = os.path.join(root, f)
        if f.lower().endswith('.jpg'):
            img_paths.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
assert meta_path is not None, '找不到 HAM10000_metadata.csv'
df = pd.read_csv(meta_path)
df['path'] = df['image_id'].map(img_paths)
assert df['path'].notna().all(), '有影像找不到檔案'
df = df.sort_values('image_id').reset_index(drop=True)
print(f'影像 {len(df):,} 張，病灶 {df.lesion_id.nunique():,} 個')
SMOKE_MODE = SMOKE_TEST
# 00 以 pHash 候選加 SSIM ≥ 0.90 找到、並經人工確認為同一病灶但病灶編號不同的影像。
# 切分前把它們視為同一個病灶，避免同一病灶被分到訓練集與測試集。原始編號保留在 lesion_id_original
LESION_MERGE = [('HAM_0004919', 'HAM_0000140')]
df['lesion_id_original'] = df['lesion_id']
if not SMOKE_MODE:
    for keep_id, merge_id in LESION_MERGE:
        assert {keep_id, merge_id}.issubset(set(df['lesion_id'])), f'找不到病灶編號 {keep_id} 或 {merge_id}'
        df.loc[df['lesion_id'] == merge_id, 'lesion_id'] = keep_id
    conf = os.path.join(CHECK_DIR, 'ham_confirmed_same_photo.csv')
    assert os.path.exists(conf), '找不到 ham_confirmed_same_photo.csv，請先用正式版 00 執行資料檢查'
    c_ = pd.read_csv(conf)
    found = {frozenset(x) for x in zip(c_['lesion_a'], c_['lesion_b'])}
    assert found == {frozenset(x) for x in LESION_MERGE}, f'00 確認的重複病灶與 LESION_MERGE 不一致：{found}。請人工確認後更新 LESION_MERGE'
print('合併後病灶數：', df['lesion_id'].nunique())
y = df['dx'].map(CLS2IDX).values

# E4 外部資料：訓練時預測全部 1,705 張，排除名單在分析時才套用
ext = None
if RUN_E4:
    ext_file = os.path.join(CHECK_DIR, 'isic2018_external.csv')
    assert os.path.exists(ext_file), '找不到 isic2018_external.csv，請先執行 00 筆記本，或把 RUN_E4 改成 False'
    ext = pd.read_csv(ext_file)
    assert {'exclude', 'exclude_sensitivity'}.issubset(ext.columns), 'isic2018_external.csv 是舊版，請用正式版 00 重新執行'
    assert ext['image_id'].is_unique and (len(ext) == 1705 or SMOKE_TEST), 'ISIC 2018 影像數不是 1,705 張'
    y_ext = ext['dx'].map(CLS2IDX).values
    print('外部資料（全部）：', len(ext), '正式保留', int((~ext['exclude']).sum()), '張，敏感度分析保留', int((~ext['exclude_sensitivity']).sum()), '張')

## 2. 資料切分（與 Baseline 筆記本完全相同）

In [ ]:
holdout = VAL_RATIO + TEST_RATIO
test_share = TEST_RATIO / holdout
assert (df.groupby('lesion_id')['dx'].nunique() == 1).all(), '合併後有病灶對應到多種診斷'
LES_DX = df.groupby('lesion_id')['dx'].first()
LES_SIZE = df.groupby('lesion_id').size()          # 每個病灶在整個資料集的影像數

def split_by_image(seed):
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=holdout, stratify=df['dx'], random_state=seed)
    va, te = train_test_split(tmp, test_size=test_share, stratify=df['dx'].values[tmp], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(te)

def split_by_lesion(seed):
    les = df.groupby('lesion_id')['dx'].first()
    tr_l, tmp_l = train_test_split(les.index.values, test_size=holdout, stratify=les.values, random_state=seed)
    va_l, te_l = train_test_split(tmp_l, test_size=test_share, stratify=les.loc[tmp_l].values, random_state=seed)
    lid = df['lesion_id'].values
    return (np.where(np.isin(lid, tr_l))[0], np.where(np.isin(lid, va_l))[0], np.where(np.isin(lid, te_l))[0])

def control_removal(seed, tr, va, te, removed_b):
    """模型 C：從訓練集移除與測試集無關的「整個病灶」，每個類別移除的影像數與模型 B 相同。
    優先抽取在整個資料集有多張影像的病灶（模型 B 移除的病灶都屬於這種），不足時再用單張影像的病灶補足。"""
    lid, dx = df['lesion_id'].values, df['dx'].values
    rng = np.random.default_rng(seed)
    test_les, val_les = set(lid[te]), set(lid[va])
    tr_count = pd.Series(lid[tr]).value_counts()              # 每個病灶在訓練集的影像數
    target = pd.Series(dx[removed_b]).value_counts()
    chosen, report = [], []
    for c in CLASS_ORDER:
        need = int(target.get(c, 0))
        if need == 0:
            continue
        cand = sorted(l for l in tr_count.index if l not in test_les and LES_DX[l] == c)
        multi = [l for l in cand if LES_SIZE[l] >= 2]
        single = [l for l in cand if LES_SIZE[l] < 2]
        got, got_multi = 0, 0
        for l in list(rng.permutation(multi)) + list(rng.permutation(single)):
            k = int(tr_count[l])
            if got + k <= need:
                chosen.append(l); got += k
                got_multi += k if LES_SIZE[l] >= 2 else 0
            if got == need:
                break
        b_les = set(lid[removed_b][dx[removed_b] == c]); c_les = set(l for l in chosen if LES_DX[l] == c)
        report.append({'種子': seed, '類別': c, '模型 B 移除影像': need, '模型 C 移除影像': got,
                       '模型 B 移除病灶數': len(b_les), '模型 C 移除病灶數': len(c_les),
                       '模型 C 來自多張影像病灶的影像比例': round(got_multi / got, 3) if got else np.nan,
                       '模型 B 移除病灶也在驗證集的數量': len(b_les & val_les), '模型 C 移除病灶也在驗證集的數量': len(c_les & val_les)})
    removed_c = tr[np.isin(lid[tr], chosen)]
    return np.setdiff1d(tr, removed_c), pd.DataFrame(report)

CONTROL_REPORT = []
def get_splits(seed):
    tr, va, te = split_by_image(seed)
    lid = df['lesion_id'].values
    tr_noleak = tr[~np.isin(lid[tr], lid[te])]
    tr_ctrl, rep = control_removal(seed, tr, va, te, np.setdiff1d(tr, tr_noleak))
    CONTROL_REPORT.append(rep)
    # 模型 B、C 選擇回合用的驗證集：移除與測試集、模型 B 訓練集或模型 C 訓練集同病灶的影像
    # 這樣驗證集的病灶不會出現在任一方的訓練集，兩個模型的回合選擇不受訓練與驗證重疊程度不同的影響
    va_clean = va[~(np.isin(lid[va], lid[te]) | np.isin(lid[va], lid[tr_noleak]) | np.isin(lid[va], lid[tr_ctrl]))]
    return {'lesion': split_by_lesion(seed), 'image': (tr, va, te),
            'image_noleak': (tr_noleak, va_clean, te), 'image_ctrl': (tr_ctrl, va_clean, te)}

def seen_mask(seed):
    # 依影像切分的測試集中，哪些影像的病灶也出現在（模型 A 的）訓練集。模型 C 保留了這些影像
    tr, _, te = SPLITS_BY_SEED[seed]['image']
    lid = df['lesion_id'].values
    return np.isin(lid[te], lid[tr])

SPLITS_BY_SEED = {s: get_splits(s) for s in SEEDS}

# 切分指紋：結果資料夾若存有不同切分（例如合併病灶前）產生的結果，直接停止，避免新舊結果混用
import hashlib
_h = hashlib.sha256(repr(LESION_MERGE).encode())
for s in SEEDS:
    for name in ['lesion', 'image', 'image_noleak', 'image_ctrl']:
        for part in SPLITS_BY_SEED[s][name]:
            _h.update(np.asarray(part, dtype=np.int64).tobytes())
SPLIT_FINGERPRINT = _h.hexdigest()[:16]
_fp = os.path.join(RES_DIR, 'split_fingerprint.txt')
if os.path.exists(_fp):
    assert open(_fp).read().strip() == SPLIT_FINGERPRINT, f'{RES_DIR} 內的結果來自不同的資料切分。請把舊資料夾改名備份後重新執行'
else:
    assert not any(f.endswith(('.csv', '.json')) for f in os.listdir(RES_DIR)), f'{RES_DIR} 已有結果但沒有切分指紋，可能是舊版產出。請把舊資料夾改名備份後重新執行'
    open(_fp, 'w').write(SPLIT_FINGERPRINT)
print('切分指紋：', SPLIT_FINGERPRINT)
control_report = pd.concat(CONTROL_REPORT, ignore_index=True)

# 檢查：模型 B 訓練集與測試集沒有同病灶，模型 C 與 B 的類別張數相同
for s in SEEDS:
    lid = df['lesion_id'].values
    trb, _, te = SPLITS_BY_SEED[s]['image_noleak']
    trc = SPLITS_BY_SEED[s]['image_ctrl'][0]
    assert not np.isin(lid[trb], lid[te]).any(), f'種子 {s}：模型 B 仍有測試病灶'
    assert set(lid[np.setdiff1d(SPLITS_BY_SEED[s]['image'][0], trc)]).isdisjoint(set(lid[te])), f'種子 {s}：模型 C 移除到測試病灶'
    vab = SPLITS_BY_SEED[s]['image_noleak'][1]
    assert not np.isin(lid[vab], lid[te]).any(), f'種子 {s}：模型 B、C 的驗證集仍有測試病灶'
    assert not np.isin(lid[vab], lid[trb]).any() and not np.isin(lid[vab], lid[trc]).any(), f'種子 {s}：模型 B、C 的驗證集仍有訓練病灶'
    vc = pd.Series(df['dx'].values[vab]).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
    print(f'種子 {s} 模型 B、C 驗證集各類影像數：', vc.to_dict())
    assert SMOKE_MODE or (vc >= 3).all(), f'種子 {s}：模型 B、C 的驗證集有類別少於 3 張，無法可靠選擇回合'
r = control_report
bad = r[r['模型 B 移除影像'] != r['模型 C 移除影像']]
# 模型 B 與 C 每個類別的移除張數必須完全相同，否則 C − B 的比較前提不成立，停止執行
assert bad.empty, '模型 C 無法完全配對，請勿開始 E3b 訓練：\n' + bad.to_string(index=False)

# 與 Baseline 的切分名單比對
ref = os.path.join(BASELINE_DIR, 'split_assignment.csv')
if os.path.exists(ref):
    a = pd.read_csv(ref, keep_default_na=False).set_index('image_id').loc[df['image_id']]
    for s in SEEDS:
        for name in ['lesion', 'image']:
            tr, va, te = SPLITS_BY_SEED[s][name]
            col = np.full(len(df), '', dtype=object); col[tr], col[va], col[te] = 'train', 'val', 'test'
            assert (a[f'seed{s}_{name}'].values == col).all(), f'種子 {s} {name} 切分與 Baseline 不一致'
        for name in ['image_noleak', 'image_ctrl']:
            removed = np.isin(np.arange(len(df)), np.setdiff1d(SPLITS_BY_SEED[s]['image'][0], SPLITS_BY_SEED[s][name][0]))
            assert (a[f'seed{s}_{name}_removed'].astype(str).values == removed.astype(str)).all(), f'種子 {s} {name} 與 Baseline 不一致'
    print('切分與 Baseline 一致')
else:
    print('未找到 Baseline 切分檔，略過比對（程式與種子相同，結果應一致）')

## 3. 影像預先縮放（224×224，存成快取）

In [ ]:
def load_resized(path, size=IMG_SIZE):
    return np.asarray(Image.open(path).convert('RGB').resize((size, size), Image.BILINEAR), dtype=np.uint8)

def cached_images(paths, ids, name):
    f, fi = os.path.join(CACHE_DIR, f'{name}.npy'), os.path.join(CACHE_DIR, f'{name}_ids.txt')
    if os.path.exists(f) and os.path.exists(fi) and open(fi).read().split() == list(ids):
        return np.load(f, mmap_mode='r')
    t0 = time.time()
    arr = np.stack(Parallel(n_jobs=-1)(delayed(load_resized)(p) for p in paths))
    np.save(f, arr); open(fi, 'w').write('\n'.join(ids))
    print(f'{name}：縮放完成，耗時 {time.time()-t0:.0f} 秒')
    return np.load(f, mmap_mode='r')

IMAGES = cached_images(df['path'], df['image_id'], f'ham_images_{IMG_SIZE}px')
EXT_IMAGES = cached_images([os.path.join(BASE, p) for p in ext['path']], ext['image_id'], f'isic2018_all_images_{IMG_SIZE}px') if ext is not None else None
print('HAM10000：', IMAGES.shape, '外部資料：', None if EXT_IMAGES is None else EXT_IMAGES.shape)

## 4. 資料集、增強、模型（與期中相同）

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_aug = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), antialias=True),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
])
to_input = T.Compose([T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])

class SkinDataset(Dataset):
    def __init__(self, idx, augment=False, images=None, labels=None):
        self.idx, self.augment = np.asarray(idx), augment
        self.images = IMAGES if images is None else images
        self.labels = y if labels is None else labels
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = torch.from_numpy(np.array(self.images[j])).permute(2, 0, 1)
        if self.augment:
            img = train_aug(img)
        return to_input(img), int(self.labels[j])

def build_model():
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None
    m = models.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASS_ORDER))
    return m.to(DEVICE)

@torch.no_grad()
def predict(model, idx, images=None, labels=None):
    model.eval()
    loader = DataLoader(SkinDataset(idx, images=images, labels=labels), batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)
    out = []
    for xb, _ in loader:
        with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
            out.append(model(xb.to(DEVICE)).float().softmax(1).cpu())
    return torch.cat(out).numpy()

def evaluate(y_true, y_pred, labels=None):
    labels = list(range(len(CLASS_ORDER))) if labels is None else labels
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    mp, mr, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[MEL], zero_division=0)
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Precision (macro)': p, 'Recall (macro)': r,
            'F1-score (macro)': f, 'Balanced Accuracy': r,
            'Melanoma Recall': mr[0] if (y_true == MEL).any() else np.nan,          # 沒有黑色素瘤時不計算
            'Melanoma Precision': mp[0],                                          # 沒有判為黑色素瘤時記為 0
            'Melanoma FN': int(((y_true == MEL) & (y_pred != MEL)).sum()),
            'Melanoma FP': int(((y_true != MEL) & (y_pred == MEL)).sum())}

## 5. 訓練函式

提出方法為「類別權重＋資料增強」。類別權重 = 訓練影像數 ／（類別數 × 該類影像數），只依訓練集計算。

In [ ]:
CONFIGS = {
    'none':   dict(class_weight=False, augment=False, oversample=False, label='不處理'),
    'cw':     dict(class_weight=True,  augment=False, oversample=False, label='類別權重'),
    'aug':    dict(class_weight=False, augment=True,  oversample=False, label='資料增強'),
    'cw+aug': dict(class_weight=True,  augment=True,  oversample=False, label='類別權重＋資料增強（提出方法）'),
    'os+aug': dict(class_weight=False, augment=True,  oversample=True,  label='過採樣＋資料增強'),
}
MAIN = 'cw+aug'
E2_ORDER = ['none', 'cw', 'aug', 'cw+aug', 'os+aug']
RUNS_CSV = os.path.join(RES_DIR, 'all_runs.csv')

def tag(cfg, split, seed):
    return f'{cfg.replace("+", "-")}__{split}__seed{seed}'

def train_run(cfg_name, split, seed, epochs, save_dir=PRED_DIR, save_tag=None, *, pilot_only=False):
    cfg = CONFIGS[cfg_name]
    set_seed(seed)
    tr, va, te = SPLITS_BY_SEED[seed][split]
    if SMOKE_TEST:
        tr, va, te = tr[:96], va[:64], te[:64]
    counts = np.bincount(y[tr], minlength=len(CLASS_ORDER))
    g = torch.Generator().manual_seed(seed)
    ds_tr = SkinDataset(tr, augment=cfg['augment'])
    if cfg['oversample']:
        sw = 1.0 / np.maximum(counts[y[tr]], 1)
        sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(tr), replacement=True, generator=g)
        loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, drop_last=True)
    else:
        loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=NUM_WORKERS, drop_last=True)
    weight = None
    if cfg['class_weight']:
        weight = torch.tensor(len(tr) / (len(CLASS_ORDER) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight)

    model = build_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    best_f1, best_state, wait, history = -1, None, 0, []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sched.step()
        val_f1 = f1_score(y[va], predict(model, va).argmax(1), labels=list(range(len(CLASS_ORDER))), average='macro', zero_division=0)
        history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'val_macro_f1': val_f1})
        print(f'  [種子 {seed} | {cfg_name} | {split}] epoch {epoch:2d}  loss {np.mean(losses):.4f}  val macro-F1 {val_f1:.4f}')
        if val_f1 > best_f1:
            best_f1, wait = val_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:                 # 試跑也使用相同的提前停止規則
                print('  提前停止'); break
    secs = time.time() - t0
    model.load_state_dict(best_state)
    if pilot_only:
        # 回合數試跑只使用訓練/驗證集，禁止產生測試集或外部集分數與指標。
        t = save_tag or 'pilot'
        pd.DataFrame(history).to_csv(os.path.join(save_dir, f'history__{t}.csv'), index=False)
        return {'試跑種子': seed, '最佳驗證 macro-F1': float(best_f1), '訓練秒數': secs}, history

    t = save_tag or tag(cfg_name, split, seed)
    probs = {'val': predict(model, va), 'test': predict(model, te),
             'val_image_id': df.image_id.iloc[va].to_numpy(dtype=str),
             'test_image_id': df.image_id.iloc[te].to_numpy(dtype=str)}
    if EXT_IMAGES is not None:
        probs['ext'] = predict(model, np.arange(len(ext)), images=EXT_IMAGES, labels=y_ext)
        probs['ext_image_id'] = ext.image_id.to_numpy(dtype=str)
    np.savez_compressed(os.path.join(save_dir, f'prob__{t}.npz'), **probs)
    pd.DataFrame(history).to_csv(os.path.join(save_dir, f'history__{t}.csv'), index=False)
    if True:                              # 所有設定都保存最佳回合的權重
        torch.save(best_state, os.path.join(MODEL_DIR, f'model__{t}.pt'))

    row = {'實驗設定': cfg_name, '說明': cfg['label'], '切分': split, '種子': seed, '回合上限': epochs,
           '最佳回合': int(np.argmax([h['val_macro_f1'] for h in history]) + 1),
           '實際回合': len(history), '驗證集 macro-F1': best_f1, '訓練影像數': len(tr)}
    row.update(evaluate(y[te], probs['test'].argmax(1)))
    row['訓練秒數'] = round(secs, 1)
    return row, history

## 6. 回合數試跑（只看驗證集）

In [ ]:
DECISION_FILE = os.path.join(RES_DIR, 'epochs_decision.json')
if os.path.exists(DECISION_FILE):
    decision = json.load(open(DECISION_FILE))
    if decision.get('試跑僅使用訓練與驗證集') is not True or decision.get('試跑使用提前停止') is not True:
        raise RuntimeError('已有舊版回合決策紀錄，無法證明試跑與測試集隔離。請先備份舊紀錄，再用修正版重新試跑。')
else:
    pilot_dir = os.path.join(RES_DIR, 'pilot'); os.makedirs(pilot_dir, exist_ok=True)
    _, hist = train_run(MAIN, 'lesion', PILOT_SEED, PILOT_EPOCHS, save_dir=pilot_dir, save_tag='pilot', pilot_only=True)
    f1s = [h['val_macro_f1'] for h in hist]
    first = max(f1s[:DEFAULT_EPOCHS])
    later = max(f1s[DEFAULT_EPOCHS:]) if len(f1s) > DEFAULT_EPOCHS else None
    gain = None if later is None else later - first
    decision = {'試跑種子': PILOT_SEED, '試跑上限': PILOT_EPOCHS, '實際回合': len(f1s),
                '前 12 回合最佳驗證 macro-F1': round(first, 4),
                '第 12 回合後最佳驗證 macro-F1': None if later is None else round(later, 4),
                '提升幅度': None if gain is None else round(gain, 4), '門檻': PILOT_MIN_GAIN,
                '正式回合上限': PILOT_EPOCHS if (gain is not None and gain > PILOT_MIN_GAIN) else DEFAULT_EPOCHS,
                '試跑僅使用訓練與驗證集': True, '試跑使用提前停止': True}
    json.dump(decision, open(DECISION_FILE, 'w'), ensure_ascii=False, indent=1)
EPOCHS = decision['正式回合上限']
print(json.dumps(decision, ensure_ascii=False, indent=1))
print('正式實驗回合上限：', EPOCHS)

## 7. 正式實驗：5 個種子 × 8 次訓練

每個種子依序：提出方法（依病灶、依影像、模型 B、模型 C）→ E2 其他四種設定。一個種子跑完再跑下一個，萬一時間不夠，已完成的種子仍可先分析。

In [ ]:
PLAN = [(MAIN, 'lesion'), (MAIN, 'image'), (MAIN, 'image_noleak'), (MAIN, 'image_ctrl')] + [(c, 'lesion') for c in E2_ORDER if c != MAIN]

def run_if_needed(cfg, split, seed):
    done = pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=['實驗設定', '切分', '種子'])
    if ((done['實驗設定'] == cfg) & (done['切分'] == split) & (done['種子'] == seed)).any():
        t = tag(cfg, split, seed)
        p = os.path.join(PRED_DIR, f'prob__{t}.npz')
        h = os.path.join(PRED_DIR, f'history__{t}.csv')
        ck = os.path.join(MODEL_DIR, f'model__{t}.pt')
        row = done[(done['實驗設定'] == cfg) & (done['切分'] == split) & (done['種子'] == seed)].iloc[0]
        if (not os.path.exists(p) or not os.path.exists(h) or
            not os.path.exists(ck) or int(row['回合上限']) != EPOCHS):
            raise RuntimeError(f'已完成紀錄與實際檔案/回合設定不一致：{t}，請先核對，不能自動略過。')
        with np.load(p) as old:
            if not {'val_image_id','test_image_id'}.issubset(old.files):
                raise RuntimeError(f'舊版預測檔缺少影像 ID：{p}。請重新產生。')
            if ext is not None and ('ext_image_id' not in old.files or set(old['ext_image_id']) != set(ext['image_id'])):
                raise RuntimeError(f'預測檔沒有涵蓋全部外部影像：{p}。請重新產生。')
        return
    print(f'開始：種子 {seed} | {CONFIGS[cfg]["label"]} | {SPLIT_LABEL[split]}')
    row, _ = train_run(cfg, split, seed, EPOCHS)
    pd.concat([done, pd.DataFrame([row])], ignore_index=True).to_csv(RUNS_CSV, index=False, encoding='utf-8-sig')
    print(f'完成：test macro-F1 {row["F1-score (macro)"]:.3f}，耗時 {row["訓練秒數"]:.0f} 秒\n')

for seed in SEEDS:
    for cfg, split in PLAN:
        run_if_needed(cfg, split, seed)
runs = pd.read_csv(RUNS_CSV)
print('完成訓練次數：', len(runs), '／', len(SEEDS) * len(PLAN))

## 8. 彙總工具

In [ ]:
def fmt(s):
    s = pd.Series(s, dtype=float).dropna()
    return f'{s.mean():.3f} ± {s.std(ddof=1):.3f}' if len(s) > 1 else (f'{s.mean():.3f}' if len(s) else '')

def paired(a, b, prefix=''):
    # 只作描述：各種子差值、平均 ± 標準差、差值為正的種子數。不據此宣稱「相同」或「顯著」
    d = (pd.Series(a, dtype=float) - pd.Series(b, dtype=float)).dropna()
    return {f'{prefix}差值（平均 ± 標準差）': fmt(d), f'{prefix}差值為正的種子數': f'{int((d > 0).sum())}/{len(d)}',
            f'{prefix}各種子差值': '、'.join(f'{v:+.3f}' for v in d)}

def per_seed(cfg, split, metric):
    return runs[(runs['實驗設定'] == cfg) & (runs['切分'] == split)].set_index('種子')[metric].reindex(SEEDS)

def load_prob(cfg, split, seed):
    return np.load(os.path.join(PRED_DIR, f'prob__{tag(cfg, split, seed)}.npz'))

def test_idx(split, seed):
    te = SPLITS_BY_SEED[seed][split][2]
    return te[:64] if SMOKE_TEST else te

def val_idx(split, seed):
    va = SPLITS_BY_SEED[seed][split][1]
    return va[:64] if SMOKE_TEST else va

def prob_for(cfg, split, seed, part, ids):
    # 依影像編號取出預測機率，part 為 'val'、'test' 或 'ext'，編號不符直接報錯
    p = load_prob(cfg, split, seed)
    pos = pd.Series(np.arange(len(p[f'{part}_image_id'])), index=p[f'{part}_image_id'])
    missing = set(ids) - set(pos.index)
    assert not missing, f'預測檔缺少 {len(missing)} 張影像：{cfg} {split} {seed} {part}'
    return p[part][pos.loc[list(ids)].values]

def test_prob(cfg, split, seed):
    return prob_for(cfg, split, seed, 'test', df['image_id'].values[test_idx(split, seed)])

METRICS = ['Accuracy', 'F1-score (macro)', 'Balanced Accuracy', 'Melanoma Recall', 'Melanoma Precision']
base_runs = pd.read_csv(os.path.join(BASELINE_DIR, 'all_runs.csv')) if os.path.exists(os.path.join(BASELINE_DIR, 'all_runs.csv')) else None
def base_seed(model, split, metric):
    return base_runs[(base_runs['模型'] == model) & (base_runs['切分'] == split)].set_index('種子')[metric].reindex(SEEDS)

## 9. E1（RQ1）：表 4-1，依病灶切分，五個種子平均值 ± 標準差

In [ ]:
rows = []
if base_runs is not None:
    for m in ['SVM', 'Random Forest']:
        rows.append({'模型': m, **{k: fmt(base_seed(m, 'lesion', k)) for k in METRICS}})
rows.append({'模型': 'EfficientNet-B0（提出方法）', **{k: fmt(per_seed(MAIN, 'lesion', k)) for k in METRICS}})
table41 = pd.DataFrame(rows)
table41.to_csv(os.path.join(RES_DIR, 'table4-1_E1_mean_sd.csv'), index=False, encoding='utf-8-sig')
print(table41.to_string(index=False))

if base_runs is not None:
    e1p = []
    for m in ['SVM', 'Random Forest']:
        for k in ['F1-score (macro)', 'Melanoma Recall']:
            e1p.append({'比較': f'EfficientNet-B0 − {m}', '指標': k, **paired(per_seed(MAIN, 'lesion', k), base_seed(m, 'lesion', k))})
    e1p = pd.DataFrame(e1p)
    e1p.to_csv(os.path.join(RES_DIR, 'E1_paired_diff.csv'), index=False, encoding='utf-8-sig')
    print(e1p.to_string(index=False))

## 10. E2（RQ2）：五種不平衡處理

除了 macro-F1 與黑色素瘤召回率，另列黑色素瘤精確率與平均精確率（AP，不受閾值影響）。

In [ ]:
def mel_ap(cfg, seed):
    p = test_prob(cfg, 'lesion', seed)
    return average_precision_score(y[test_idx('lesion', seed)] == MEL, p[:, MEL])

e2 = []
for c in E2_ORDER:
    r = {'設定': CONFIGS[c]['label']}
    r.update({k: fmt(per_seed(c, 'lesion', k)) for k in ['F1-score (macro)', 'Balanced Accuracy', 'Melanoma Recall', 'Melanoma Precision']})
    r['黑色素瘤 AP'] = fmt([mel_ap(c, s) for s in SEEDS])
    r['最佳回合'] = fmt(per_seed(c, 'lesion', '最佳回合'))
    e2.append(r)
e2 = pd.DataFrame(e2)
e2.to_csv(os.path.join(RES_DIR, 'E2_mean_sd.csv'), index=False, encoding='utf-8-sig')
print(e2.to_string(index=False))

# 配對比較：各設定 − 不處理，以及提出方法 − 類別權重
e2p = []
ap = {c: pd.Series({s: mel_ap(c, s) for s in SEEDS}) for c in E2_ORDER}
pairs_ = [(c, 'none') for c in E2_ORDER if c != 'none'] + [(MAIN, 'cw')]
for a, b in pairs_:
    for k in ['F1-score (macro)', 'Melanoma Recall', 'Melanoma Precision']:
        e2p.append({'比較': f'{CONFIGS[a]["label"]} − {CONFIGS[b]["label"]}', '指標': k, **paired(per_seed(a, 'lesion', k), per_seed(b, 'lesion', k))})
    e2p.append({'比較': f'{CONFIGS[a]["label"]} − {CONFIGS[b]["label"]}', '指標': '黑色素瘤 AP', **paired(ap[a], ap[b])})
e2p = pd.DataFrame(e2p)
e2p.to_csv(os.path.join(RES_DIR, 'E2_paired_diff.csv'), index=False, encoding='utf-8-sig')
print(e2p.to_string(index=False))

### 黑色素瘤決策閾值（只用驗證集選擇）

規則：黑色素瘤機率 ≥ 閾值就判為黑色素瘤，否則判為其餘類別中機率最高者。閾值在 0.01–0.99（間隔 0.01）中，選驗證集黑色素瘤 F2 分數最高者，再到測試集回報調整前後的結果。

In [ ]:
GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)

def apply_threshold(prob, t):
    other = prob.copy(); other[:, MEL] = -1
    return np.where(prob[:, MEL] >= t, MEL, other.argmax(1))

def f2(y_true, y_pred):
    tp = ((y_true == MEL) & (y_pred == MEL)).sum()
    fp = ((y_true != MEL) & (y_pred == MEL)).sum()
    fn = ((y_true == MEL) & (y_pred != MEL)).sum()
    return 5 * tp / (5 * tp + 4 * fn + fp) if tp > 0 else 0.0

thr_long = []
for c in E2_ORDER:
    for s in SEEDS:
        va, te = val_idx('lesion', s), test_idx('lesion', s)
        p_val = prob_for(c, 'lesion', s, 'val', df['image_id'].values[va])
        p_test = test_prob(c, 'lesion', s)
        scores = [f2(y[va], apply_threshold(p_val, t)) for t in GRID]
        t_best = float(GRID[int(np.argmax(scores))])
        before = evaluate(y[te], p_test.argmax(1))
        after = evaluate(y[te], apply_threshold(p_test, t_best))
        thr_long.append({'設定': c, '種子': s, '閾值': t_best,
                         **{f'{k}（調整前）': before[k] for k in ['Melanoma Recall', 'Melanoma Precision', 'Melanoma FN', 'Melanoma FP', 'F1-score (macro)']},
                         **{f'{k}（調整後）': after[k] for k in ['Melanoma Recall', 'Melanoma Precision', 'Melanoma FN', 'Melanoma FP', 'F1-score (macro)']}})
thr_long = pd.DataFrame(thr_long)
thr_long.to_csv(os.path.join(RES_DIR, 'E2_threshold_per_seed.csv'), index=False, encoding='utf-8-sig')
thr = thr_long.groupby('設定', sort=False).agg(lambda s: fmt(s)).drop(columns='種子')
thr.index = [CONFIGS[c]['label'] for c in thr.index]
thr.to_csv(os.path.join(RES_DIR, 'E2_threshold_mean_sd.csv'), encoding='utf-8-sig')
print(thr.T.to_string())

In [ ]:
# 圖：E2 五種設定（誤差線為五個種子的標準差）
labels_en = {'none': 'None', 'cw': 'Class weight', 'aug': 'Augmentation', 'cw+aug': 'Weight + Aug\n(proposed)', 'os+aug': 'Oversample + Aug'}
xs = np.arange(len(E2_ORDER)); w = 0.27
fig, ax = plt.subplots(figsize=(9, 4))
for k, (metric, color, name) in enumerate([('F1-score (macro)', '#4C72B0', 'Macro F1'), ('Melanoma Recall', '#C44E52', 'Melanoma recall'), ('Melanoma Precision', '#55A868', 'Melanoma precision')]):
    m_ = [per_seed(c, 'lesion', metric).mean() for c in E2_ORDER]
    s_ = [per_seed(c, 'lesion', metric).std(ddof=1) for c in E2_ORDER]
    ax.bar(xs + (k - 1) * w, m_, w, yerr=s_, capsize=3, label=name, color=color)
ax.set_xticks(xs); ax.set_xticklabels([labels_en[c] for c in E2_ORDER], fontsize=9)
ax.set_ylim(0, 1); ax.set_ylabel('Test score (mean ± SD, 5 seeds)')
ax.set_title('E2: Effect of class-imbalance handling (EfficientNet-B0)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig_E2_mean_sd.png'), dpi=300); plt.show()

## 11. E3（RQ3）：依影像切分 − 依病灶切分（同種子配對）

In [ ]:
e3 = []
for k in ['F1-score (macro)', 'Melanoma Recall', 'Balanced Accuracy']:
    e3.append({'模型': 'EfficientNet-B0', '指標': k, '依病灶切分': fmt(per_seed(MAIN, 'lesion', k)),
               '依影像切分': fmt(per_seed(MAIN, 'image', k)), **paired(per_seed(MAIN, 'image', k), per_seed(MAIN, 'lesion', k))})
e3 = pd.DataFrame(e3)
base_e3 = os.path.join(BASELINE_DIR, 'E3_baseline_paired.csv')
if os.path.exists(base_e3):
    e3 = pd.concat([pd.read_csv(base_e3, keep_default_na=False), e3], ignore_index=True)
e3.to_csv(os.path.join(RES_DIR, 'E3_all_paired.csv'), index=False, encoding='utf-8-sig')
print(e3.to_string(index=False))

## 12. E3b（RQ3）：同一份測試集比較模型 A、B、C

- **主要比較 C − B**：兩者訓練張數與類別組成相同，主要差異在是否含有與測試影像同病灶的影像，移除的病灶數與影像難度仍可能不同。
- **輔助比較 A − B**：原始依影像切分與移除重疊後的差距，含有訓練量減少的影響。
- **分群**：依測試影像的病灶是否出現在模型 A 的訓練集分兩群。若差距來自資料洩漏，C − B 應主要出現在「病灶出現過」的一群。兩群的 macro-F1 只用兩群都有的類別計算。
- 模型 B、C 選擇回合用的驗證集已移除與測試集或兩者訓練集同病灶的影像，模型 A 沿用 E3 的原驗證集，僅作參考。
- C − B 是控制訓練影像數與類別組成後的比較，B、C 移除的病灶組成仍可能不同（見 `E3b_control_matching.csv`），不視為純粹的因果效果。

In [ ]:
E3B_MODELS = {'A': 'image', 'B': 'image_noleak', 'C': 'image_ctrl'}
E3B_METRICS = ['F1-score (macro)', 'Melanoma Recall', 'Accuracy']
ALL_LABELS = list(range(len(CLASS_ORDER)))

e3b_long = []
for s in SEEDS:
    te = test_idx('image', s)
    yt_all = y[te]
    seen = seen_mask(s)[:len(te)]
    common = sorted(set(yt_all[seen]) & set(yt_all[~seen]))
    preds = {k: test_prob(MAIN, sp, s).argmax(1) for k, sp in E3B_MODELS.items()}
    for group, mask, labels in [('全部', np.ones(len(te), bool), ALL_LABELS), ('病灶出現過', seen, common), ('病灶未出現過', ~seen, common)]:
        if mask.sum() == 0 or not labels:
            continue
        res = {k: evaluate(yt_all[mask], p[mask], labels=labels) for k, p in preds.items()}
        for metric in E3B_METRICS:
            e3b_long.append({'模型': 'EfficientNet-B0', '群組': group, '種子': s, '指標': metric, '影像數': int(mask.sum()),
                             '計算類別數': len(labels), '計算類別': '、'.join(CLASS_ORDER[i] for i in labels), **{k: res[k][metric] for k in E3B_MODELS}})
e3b_long = pd.DataFrame(e3b_long)
e3b_long.to_csv(os.path.join(RES_DIR, 'E3b_cnn_per_seed.csv'), index=False, encoding='utf-8-sig')

rows = []
for (m, g, metric), d in e3b_long.groupby(['模型', '群組', '指標'], sort=False):
    d = d.set_index('種子')
    rows.append({'模型': m, '群組': g, '指標': metric, '影像數（平均）': round(d['影像數'].mean()),
                 '計算類別數': int(d['計算類別數'].min()),
                 '模型 A': fmt(d['A']), '模型 B': fmt(d['B']), '模型 C': fmt(d['C']),
                 **paired(d['C'], d['B'], 'C−B '), **paired(d['A'], d['B'], 'A−B ')})
e3b = pd.DataFrame(rows)
base_e3b = os.path.join(BASELINE_DIR, 'E3b_baseline_summary.csv')
if os.path.exists(base_e3b):
    e3b = pd.concat([pd.read_csv(base_e3b, keep_default_na=False), e3b], ignore_index=True)
e3b.to_csv(os.path.join(RES_DIR, 'E3b_all_summary.csv'), index=False, encoding='utf-8-sig')
control_report.to_csv(os.path.join(RES_DIR, 'E3b_control_matching.csv'), index=False, encoding='utf-8-sig')
print('模型 B、C 移除的訓練影像數：',
      {s: (len(SPLITS_BY_SEED[s]['image'][0]) - len(SPLITS_BY_SEED[s]['image_noleak'][0]),
           len(SPLITS_BY_SEED[s]['image'][0]) - len(SPLITS_BY_SEED[s]['image_ctrl'][0])) for s in SEEDS})
print(e3b.to_string(index=False))

In [ ]:
# 圖：E3b 分群（macro-F1），三個模型
d = e3b_long[(e3b_long['指標'] == 'F1-score (macro)') & (e3b_long['群組'] != '全部')]
grp = ['病灶出現過', '病灶未出現過']; grp_en = ['Lesion seen in training', 'Lesion not seen']
xs = np.arange(2); w = 0.26
fig, ax = plt.subplots(figsize=(7, 4))
for k, (col, name, color) in enumerate([('A', 'Model A (original training set)', '#DD8452'),
                                        ('C', 'Model C (unrelated lesions removed)', '#C44E52'),
                                        ('B', 'Model B (overlapping lesions removed)', '#4C72B0')]):
    m_ = [d[d['群組'] == g][col].mean() for g in grp]; s_ = [d[d['群組'] == g][col].std(ddof=1) for g in grp]
    ax.bar(xs + (k - 1) * w, m_, w, yerr=s_, capsize=3, label=name, color=color)
ax.set_xticks(xs); ax.set_xticklabels(grp_en); ax.set_ylim(0, 1)
ax.set_ylabel('Macro F1 (mean ± SD, 5 seeds)'); ax.set_title('E3b: Same image-level test set')
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig_E3b_subgroups.png'), dpi=300); plt.show()

## 13. E4：外部資料（ISIC 2018）

- **正式結果**：只排除影像編號相同、IL 病灶編號相同，或 SSIM 確認為同一張照片的影像。
- **敏感度分析**：另外排除所有 pHash 候選。
- 若 00 的稽核未通過，這一節會跳過。

In [ ]:
E4_SETS = {'正式': 'exclude', '敏感度分析': 'exclude_sensitivity'}
audit = json.load(open(os.path.join(CHECK_DIR, 'e4_audit_status.json'), encoding='utf-8')) if ext is not None else {}
if ext is not None and audit.get('e4_ready_for_provisional_evaluation') is True:
    rows = []
    for set_name, col in E4_SETS.items():
        keep = ~ext[col].values
        ids = ext['image_id'].values[keep]
        srcs = ext['source'].values[keep]
        groups = [('全部', np.ones(keep.sum(), bool))]
        if len(set(srcs)) > 1:
            groups += [(g, srcs == g) for g in ['same', 'different'] if (srcs == g).any()]
        for sp in ['lesion', 'image']:
            preds = [prob_for(MAIN, sp, seed, 'ext', ids).argmax(1) for seed in SEEDS]
            for g, mask in groups:
                yt = y_ext[keep][mask]
                labels = ALL_LABELS                      # 固定七類，兩種排除方案才能直接比較
                vals = [evaluate(yt, p[mask], labels=labels) for p in preds]
                rows.append({'資料集': set_name, '模型': 'EfficientNet-B0', '訓練切分': sp, '外部資料群組': g, '影像數': int(mask.sum()),
                             '計算類別數': len(labels),
                             **{k: fmt([v[k] for v in vals]) for k in ['F1-score (macro)', 'Balanced Accuracy', 'Melanoma Recall']}})
    e4 = pd.DataFrame(rows)
    base_e4 = os.path.join(BASELINE_DIR, 'E4_baseline_external.csv')
    if os.path.exists(base_e4):
        e4 = pd.concat([pd.read_csv(base_e4, keep_default_na=False), e4], ignore_index=True)
    e4.to_csv(os.path.join(RES_DIR, 'E4_all_external.csv'), index=False, encoding='utf-8-sig')
    print(e4.to_string(index=False))
elif ext is not None:
    print('⚠ E4 稽核未通過，略過 E4 分析。請先檢查 00 的 e4_audit_status.json')

## 14. 學習曲線與混淆矩陣（提出方法、依病灶切分）

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for s in SEEDS:
    h = pd.read_csv(os.path.join(PRED_DIR, f'history__{tag(MAIN, "lesion", s)}.csv'))
    ax.plot(h['epoch'], h['val_macro_f1'], 'o-', ms=3, label=f'seed {s}')
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation macro F1'); ax.legend(fontsize=8)
ax.set_title('Validation macro F1 by epoch (EfficientNet-B0, lesion-level split)')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig_learning_curves.png'), dpi=300); plt.show()

cm = sum(confusion_matrix(y[test_idx('lesion', s)], test_prob(MAIN, 'lesion', s).argmax(1), labels=range(len(CLASS_ORDER))) for s in SEEDS)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_ORDER).plot(ax=axes[0], cmap='Blues', colorbar=False, values_format='d')
ConfusionMatrixDisplay(cm / np.maximum(cm.sum(1, keepdims=True), 1), display_labels=CLASS_ORDER).plot(ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f')
for ax, t in zip(axes, ['Counts (5 seeds pooled)', 'Normalized by true class']):
    ax.set_title(f'EfficientNet-B0 — {t}'); ax.set_xlabel('Predicted class'); ax.set_ylabel('True class')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig4-1_cnn_confusion_matrix.png'), dpi=300); plt.show()

## 15. Grad-CAM（種子 42 的提出方法）

上排為判斷正確的黑色素瘤（熱圖對應黑色素瘤類別），下排為漏判案例（熱圖對應模型誤判的類別）。

In [ ]:
gc_seed = SEEDS[0]
model = build_model()
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'model__{tag(MAIN, "lesion", gc_seed)}.pt'), map_location=DEVICE))
model.eval()

def grad_cam(model, x, class_idx):
    acts, grads = {}, {}
    layer = model.features[-1]
    def fwd_hook(m, i, o):
        acts['a'] = o
        o.register_hook(lambda g: grads.__setitem__('g', g))
    h = layer.register_forward_hook(fwd_hook)
    model.zero_grad()
    out = model(x)
    out[0, class_idx].backward()
    h.remove()
    w = grads['g'].mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * acts['a']).sum(1)).squeeze().detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return np.asarray(Image.fromarray(np.uint8(cam * 255)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0

te = test_idx('lesion', gc_seed)
pred = test_prob(MAIN, 'lesion', gc_seed).argmax(1)
is_mel = y[te] == MEL
rng = np.random.default_rng(gc_seed)
correct = rng.permutation(te[is_mel & (pred == MEL)])[:4]
missed_mask = is_mel & (pred != MEL)
missed = rng.permutation(te[missed_mask])[:4]
missed_pred = dict(zip(te[missed_mask], pred[missed_mask]))

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax in axes.ravel():
    ax.axis('off')
for r, (name, ids) in enumerate([('Correct', correct), ('Missed', missed)]):
    for c, j in enumerate(ids):
        x = SkinDataset([j])[0][0].unsqueeze(0).to(DEVICE)
        target = MEL if name == 'Correct' else int(missed_pred[j])
        cam = grad_cam(model, x, target)
        axes[r, c].imshow(np.array(IMAGES[j])); axes[r, c].imshow(cam, cmap='jet', alpha=0.4)
        axes[r, c].set_title(f"{df.image_id[j]}\n" + ('Correct (CAM: mel)' if name == 'Correct' else f'Missed → {CLASS_ORDER[target]} (CAM: {CLASS_ORDER[target]})'), fontsize=8)
plt.suptitle(f'Grad-CAM for melanoma, seed {gc_seed} (top: correct, bottom: missed)')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig_gradcam_melanoma.png'), dpi=200); plt.show()

## 16. 實驗環境（表 3-5）與打包

In [ ]:
env = {'執行平台': 'Google Colab' if IN_COLAB else platform.platform(),
       '硬體': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else f'CPU（{os.cpu_count()} 核心）',
       '程式語言': 'Python ' + platform.python_version(),
       '主要套件與版本': f'PyTorch {torch.__version__}、torchvision {torchvision.__version__}、scikit-learn {sklearn.__version__}、NumPy {np.__version__}、pandas {pd.__version__}',
       '隨機種子': '、'.join(map(str, SEEDS)),
       '正式回合上限': EPOCHS}
pd.Series(env).to_csv(os.path.join(RES_DIR, 'environment_cnn.csv'), header=['規格'], encoding='utf-8-sig')
print(pd.Series(env).to_string())

pack = os.path.join(BASE, 'results_final_for_report')
os.makedirs(pack, exist_ok=True)
for d_ in [RES_DIR, BASELINE_DIR, CHECK_DIR]:
    if os.path.isdir(d_):
        for f in os.listdir(d_):
            if f.endswith(('.csv', '.png', '.json')) and not f.endswith('phash.csv'):
                shutil.copy(os.path.join(d_, f), pack)
shutil.make_archive(pack, 'zip', pack)
print('已打包：', pack + '.zip')

## 17. 輸出檔案對照報告

| 檔案 | 用在報告哪裡 |
|---|---|
| `epochs_decision.json` | 參之五 回合數試跑結果 |
| `table4-1_E1_mean_sd.csv`、`E1_paired_diff.csv` | 表 4-1、RQ1 |
| `E2_mean_sd.csv`、`E2_paired_diff.csv`、`fig_E2_mean_sd.png` | E2、RQ2 |
| `E2_threshold_mean_sd.csv`、`E2_threshold_per_seed.csv` | 閾值分析、RQ2 |
| `E3_all_paired.csv` | E3、RQ3（含 Baseline） |
| `E3b_all_summary.csv`、`fig_E3b_subgroups.png`、`E3b_control_matching.csv` | E3b、RQ3（含 Baseline 與模型 C 配對紀錄） |
| `E4_all_external.csv` | E4 外部驗證，正式與敏感度分析（含 Baseline） |
| `fig_learning_curves.png`、`fig4-1_cnn_confusion_matrix.png`、`fig_gradcam_melanoma.png` | 第肆章圖表 |
| `models/model__cw-aug__lesion__seed42.pt` | 系統原型使用的模型 |
| `all_runs.csv`、`predictions/` | 完整紀錄，上傳 GitHub |

數字以這些輸出檔為準。GPU 運算即使固定種子，重跑仍可能有些微差異。